# debug_syntax_deriver_db_3

# Add root to sys.path

In [4]:
import os
import sys

# Add the root directory to the Python path
module_dir = os.path.abspath('..')
if module_dir not in sys.path:
    sys.path.append(module_dir)
for x in sys.path:
    print(x)

/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/jupyter_debug
/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev
/opt/anaconda3/envs/python312/lib/python312.zip
/opt/anaconda3/envs/python312/lib/python3.12
/opt/anaconda3/envs/python312/lib/python3.12/lib-dynload

/opt/anaconda3/envs/python312/lib/python3.12/site-packages
/opt/anaconda3/envs/python312/lib/python3.12/site-packages/setuptools/_vendor
/Users/hale/PycharmProjects/MathAssertGPT


# Imports

In [5]:
import os

from pathlib import Path

from source.shared import AssertDB
from source.shared import SyntaxDeriver

In [6]:
def reset_syntax_deriver_db(syntax_deriver):
    syntax_deriver.syntax_deriver_db.conn.execute('DELETE FROM rule_errors;')
    syntax_deriver.syntax_deriver_db.conn.execute('DELETE FROM math_statements;')

In [7]:
corpus_folder_path = os.fspath(Path('../corpus').resolve())
print(f'corpus_folder_path: {corpus_folder_path}')

corpus_folder_path: /Users/hale/PycharmProjects/MathAssertGPT/corpus


In [8]:
assert_db = AssertDB(assert_db_file_path=Path(corpus_folder_path).joinpath('assert.db'))
syntax_deriver = SyntaxDeriver(assert_db=assert_db)

In [9]:
reset_syntax_deriver_db(syntax_deriver)
statement = r"( Fun `' F = dom F -> ( `' A |` ran F ) )"
statement = r"( ( R V , B ) /\ ~P A ( x e. ( C e. S /\ x e. W ) /\ y e. _V ) -> ( ( A , B ) -> ( <. x e. ran E. y e. A E e. B ) )"
statement = r"( <. x , y >. e. ( A e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )"
syntax_deriver.derive_syntax(statement=statement, context=None)
sql = 'SELECT id, statement, context, derivation, derivation_correct_count, syntax_deriver_error FROM math_statements ORDER BY id'
math_statement_rows = syntax_deriver.syntax_deriver_db.conn.execute(sql).fetchall()
math_statement_row = math_statement_rows[0]
sql = f'SELECT statement_id, rule_name, rule, mark_index, rule_tokens, current_rule_tokens FROM rule_errors WHERE statement_id = {math_statement_row.id} ORDER BY id'
rule_error_rows = syntax_deriver.syntax_deriver_db.conn.execute(sql).fetchall()
print(math_statement_rows)
print(math_statement_rows[0].statement)
print(math_statement_rows[0].context)
print(math_statement_rows[0].derivation)
for rule_error_row in rule_error_rows:
    print(rule_error_row)

[Row(id=1, statement='( <. x , y >. e. ( A e. _V /\\ E. y e. A ph ) -> ( ps /\\ E. y e. _V ph ) )', context=None, derivation=None, derivation_correct_count=9, syntax_deriver_error='SyntaxDeriverWffRuleError')]
( <. x , y >. e. ( A e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )
None
None
Row(statement_id=1, rule_name='cv', rule='x', mark_index=0, rule_tokens='e. _V /\\ E. y e. A ph ) -> ( ps /\\ E. y e. _V ph ) )', current_rule_tokens='e. _V /\\ E. y e. A ph ) -> ( ps /\\ E. y e. _V ph ) )')
Row(statement_id=1, rule_name='cvv', rule='_V', mark_index=0, rule_tokens='e. _V /\\ E. y e. A ph ) -> ( ps /\\ E. y e. _V ph ) )', current_rule_tokens='e. _V /\\ E. y e. A ph ) -> ( ps /\\ E. y e. _V ph ) )')
Row(statement_id=1, rule_name='cop', rule='<. A , B >.', mark_index=0, rule_tokens='e. _V /\\ E. y e. A ph ) -> ( ps /\\ E. y e. _V ph ) )', current_rule_tokens='e. _V /\\ E. y e. A ph ) -> ( ps /\\ E. y e. _V ph ) )')
Row(statement_id=1, rule_name='cotp', rule='<. A , B , C >.', mark_in

In [10]:
from source.shared import SyntaxDeriverValidationReporter

block_size = 150
syntax_deriver_validation_reporter = SyntaxDeriverValidationReporter(syntax_deriver_db=syntax_deriver.syntax_deriver_db, block_size=block_size)
syntax_deriver_validation_reporter.print_math_statement_error(example=0, error_count=1, math_statement_row=math_statement_row, print_context=False)
syntax_deriver_validation_reporter.print_rule_errors(math_statement_row=math_statement_row, rule_error_rows=rule_error_rows)

===== Example 1 error:1 =====
SyntaxDeriverWffRuleError
derivation_correct_count=9
cv: x mark_index=0 token_count=20 current_token_count=20
	full_statement: ( <. x , y >. e. ( A e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )
	statement_part: ( <. x , y >. e. ( A
	statement_rest: e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )
	statement_peek: 
	expected: x got: e.
cvv: _V mark_index=0 token_count=20 current_token_count=20
	full_statement: ( <. x , y >. e. ( A e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )
	statement_part: ( <. x , y >. e. ( A
	statement_rest: e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )
	statement_peek: 
	expected: _V got: e.
cop: <. A , B >. mark_index=0 token_count=20 current_token_count=20
	full_statement: ( <. x , y >. e. ( A e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )
	statement_part: ( <. x , y >. e. ( A
	statement_rest: e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )
	statement_peek: 
	expected: <. got: e.
cotp: <. A , B , 

# Dec 5, 2025

In [11]:
from source.shared import check_statement

In [12]:
%%time
statement = r"( Fun `' F = dom F -> ( `' A |` ran F ) )"
statement = r"( ( R V , B ) /\ ~P A ( x e. ( C e. S /\ x e. W ) /\ y e. _V ) -> ( ( A , B ) -> ( <. x e. ran E. y e. A E e. B ) )"
statement = r"( <. x , y >. e. ( A e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )"
check_statement(statement=statement, corpus_folder_path=corpus_folder_path)

----- Check -----
Statement has an error.
statement: ( <. x , y >. e. ( A e. _V /\ E. y e. A ph ) -> ( ps /\ E. y e. _V ph ) )
syntax_deriver_error: SyntaxDeriverWffRuleError
correct portion: ( <. x , y >. e. ( A
invalid_token: e.
derivation_correct_count: 9
CPU times: user 14.3 ms, sys: 2.04 ms, total: 16.4 ms
Wall time: 15 ms
